# Assesment notebook 

In [1]:
# Numerical arrays.
import numpy as np
# Numerical arrays.
import hashlib

## Problem 0: GitHub Issues

https://github.com/DmtSenk/Computational-Theory/issues

## Problem 1: Representing SHA-256 Data

Problem 1 involves investigating how the input, output, and intermediate data can be represented in Python.

|Data|Representation|Why|
|-|-|-|
|32-bit words|?|?|
|Sequences of 32-bit words|?|?|
|Input messages|?|?|
|512-bit message blocks|?|?|
|The final 256-bit hash value|?|?|
|Big-endian integers|?|?|


32-bit words

SHA-256 uses 32-bit words operating with $2^{32}$ ( from 0 up to 4,294,967,295 ).

When sum go past max number it will go back to 0.

np.unit(32) does that automatically, normal integer will just grow bigger.  
    

In [2]:
word = np.uint32(255)
print(word, type(word))

print(np.iinfo(np.uint32).max)  # Max value for uint32
print(4294967295+1) # Normal int will just grow bigger

test = np.array([4294967295], dtype=np.uint32)
print(test+1) # This will round back to 0 in uint32


255 <class 'numpy.uint32'>
4294967295
4294967296
[0]


Sequences of 32-bit words

For a sequence of 32-bit word will be used an array with data type np.uint32 ( dtype=np.uint32 ).

Using normal python array goes over 32 bits

In [3]:
words = np.array([1,1,1], dtype=np.uint32)
print(words)

print(words.dtype)
print(len(words))

[1 1 1]
uint32
3


Input messages

SHA-256 processes a message as a sequence of bits.

A message can be represented as a Python str( string ) when it is text. 

SHA-256 works with bytes, so the string should first be converted to bytes using UTF-8 encoding.

Using bytes is appropriate because SHA-256 processes the message as binary data. 

It also makes it easy to find the message length in bytes using len(). 

The length in bits can then be found by multiplying the number of bytes by 8.


In [4]:
message = "abc".encode("utf-8")
print(message)
print(len(message))       # Length in bytes.
print(len(message) * 8)   # Length in bits.

b'abc'
3
24


512-bit message blocks
I will store each SHA-256 block as an array of 16 np.uint32 words.
SHA-256 divides the message into blocks of 512 bits. Since 512 bits = 64 bytes, and each word is 4 bytes, each block contains 16 words.

In [5]:
raw = bytes(range(64))
block = np.frombuffer(raw, dtype=">u4").astype(np.uint32)

np.frombuffer converts the 64 bytes into 16 words. The >u4 means each word is a 4-byte unsigned integer in big-endian order. 
This matches the way SHA-256 represents its 512-bit blocks.

In [6]:
print(len(raw) * 8)  # 512 bits
print(len(block))    # 16 words
print(block[0])      # First 4 bytes as one number

512
16
66051


The 256-bit hash value

I will store the hash as 8 np.uint32 words, and show it as 64 hex characters.

256 bits is 8 words of 32 bits, and also 32 bytes.
Written in hexadecimal, each byte takes 2 characters, so the hash is 64 characters.
hashlib gives the hash in these forms, so I can compare my results with it later.

In [7]:
digest = hashlib.sha256(b"abc").digest()                      # 32 bytes.
hash_words = np.frombuffer(digest, dtype=">u4").astype(np.uint32)

print(len(digest))         # 32 bytes.
print(len(hash_words))     # 8 words.
print(digest.hex())        # 64 hex characters.
print(len(digest.hex()))

# The first word matches the first 8 hex characters.
print(f"{hash_words[0]:08x}", digest.hex()[:8])

32
8
ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad
64
ba7816bf ba7816bf


Big-endian integers

I will use int.from_bytes && int.to_bytes.
Biggest part of a number comes first.
SHA-256 uses this order when it turns bytes into numbers.
If the order is wrong, the number is different and the hash comes out wrong.

In [8]:
data = b"\x00\x00\x01\x00"

print(int.from_bytes(data, "big"))      # 256: correct for SHA-256.
print(int.from_bytes(data, "little"))   # 65536: wrong order, different number.
print((256).to_bytes(4, "big"))         # Number back to 4 bytes.

256
65536
b'\x00\x00\x01\x00'


## Problem 2: SHA-256 Bitwise Operations
SHA-256 uses several Boolean and bitwise functions when processing
32-bit words. In this problem, I will implement the following functions:

1. `Ch(x, y, z)`
2. `Maj(x, y, z)`
3. `Sigma0(x)`, written as $\Sigma_0^{256}(x)$ in the Secure Hash Standard
4. `Sigma1(x)`, written as $\Sigma_1^{256}(x)$ in the Secure Hash Standard
5. `sigma0(x)`, written as $\sigma_0^{256}(x)$ in the Secure Hash Standard
6. `sigma1(x)`, written as $\sigma_1^{256}(x)$ in the Secure Hash Standard

NumPy will be used so that the values involved in these operations are represented as 32-bit unsigned integers

For each function, I will:
- provide an appropriate docstring;
- explain the operation in Markdown;
- implement the function directly from its definition;
- test the function using appropriate input values.


`Ch(x, y, z)`

The SHA-256 choose function is defined as:

$$
Ch(x, y, z) = (x \mathbin{\&} y) \oplus
((\mathop{\sim}x) \mathbin{\&} z)
$$

The function examines each bit of `x`:

- if a bit of `x` is `1`, the corresponding bit is selected from `y`;
- if a bit of `x` is `0`, the corresponding bit is selected from `z`.


In [3]:
def Ch(x, y, z):
    """Select bits from y or z according to the bits of x."""
    x = np.uint32(x)
    y = np.uint32(y)
    z = np.uint32(z)

    return np.uint32((x & y) ^ (~x & z))

Testing Ch

The function is tested using the following cases:

1. If `x` is all `1` bits, the result should be `y`.
2. If `x` is all `0` bits, the result should be `z`.
3. If `x`, `y`, and `z` contain representative hexadecimal values, the
   result should match an independently calculated expected value.
4. The result should have the NumPy `uint32` data type.

In [7]:
# Test values represented as 32-bit unsigned integers.
all_zeros = np.uint32(0x00000000)
all_ones = np.uint32(0xFFFFFFFF)
x = np.uint32(0x0F0F0F0F)
y = np.uint32(0x12345678)
z = np.uint32(0x9ABCDEF0)

# If x is all ones, Ch should return y.
assert Ch(all_ones, y, z) == y

# If x is all zeros, Ch should return z.
assert Ch(all_zeros, y, z) == z

# Test the function with representative input values.
expected = np.uint32(0x92B4D6F8)
assert Ch(x, y, z) == expected

# Confirm that the result is a NumPy uint32 value.
assert isinstance(Ch(x, y, z), np.uint32)

print(f"Ch({x:#010x}, {y:#010x}, {z:#010x}) = {Ch(x, y, z):#010x}")
print("All Ch tests passed.")

Ch(0x0f0f0f0f, 0x12345678, 0x9abcdef0) = 0x92b4d6f8
All Ch tests passed.
